# Dask Exercise 2: dask delayed

* [dask delayed](https://docs.dask.org/en/stable/delayed.html)
* [dask tutorial](https://tutorial.dask.org/03_dask.delayed.html)

Skills:
* Convert a for loop into a simple dask delayed workflow 
* Get more familiar with `dask.dataframe` wrangling

In [1]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import pathlib

from dask import delayed, compute

GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/v2_rt_trips/"
                )

analysis_date = "2023-03-15"
la_metro = 182
big_blue_bus = 300
muni = 282

operators = [la_metro, big_blue_bus, muni]

## Simple Workflow to Parallelize

This is a typical workflow. 
1. Read in pandas df.
2. Apply a certain function.
3. Export df.

Let's say we have a df corresponding to each operator. We want to apply the same aggregation function and then save out the results.

Typically, we would use a loop. A loop is **sequential**. By using `dask delayed` objects, we can run those **simultaneously**. Instead of running operator 1, operator 2, operator 3, ... , operator N, why not let them run at the same time and save out the results?

There is nothing inherent in our workflow that specifies that operator 1 must be run before operator 2. We are applying the same function to each operator. To speed it up, let's use dask to run it in parallel and get our results.

In [2]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}{big_blue_bus}_{analysis_date}.parquet")

In [3]:
# Adding this just to get a summary of the df for later
df.head()

,feed_key,trip_key,gtfs_dataset_key,activity_date,trip_id,route_id,route_short_name,shape_id,direction_id,route_type,route_long_name,route_desc,calitp_itp_id,median_time,direction,mean_speed_mph,organization_name
0,4f77ef02b983eccc0869c7540f98a7d0,f088a730653c0679da62feaa9c25bc26,dbbe8ee4864a2715a40749605395d584,2023-03-15,893061,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,10:16:03.500000,Northbound,8.925867,City of Santa Monica
1,4f77ef02b983eccc0869c7540f98a7d0,8451ebd43e9159cc27b03ae2ae868ee3,dbbe8ee4864a2715a40749605395d584,2023-03-15,894836,3564,R12,26195,0,3,Venice/Westwood Sta/UCLA Rapid,None,300,14:30:33,Northbound,12.339312,City of Santa Monica
2,4f77ef02b983eccc0869c7540f98a7d0,c3ff3523997c3c43495ebdad117f95a0,dbbe8ee4864a2715a40749605395d584,2023-03-15,893098,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,16:26:50,Northbound,8.208192,City of Santa Monica
3,4f77ef02b983eccc0869c7540f98a7d0,b4d78606e25155aa02b25a26e008374e,dbbe8ee4864a2715a40749605395d584,2023-03-15,893102,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,17:06:58,Northbound,7.702262,City of Santa Monica
4,4f77ef02b983eccc0869c7540f98a7d0,12276a7ee0019ab61b04929e64f0b444,dbbe8ee4864a2715a40749605395d584,2023-03-15,893173,3554,1,26164,1,3,Main St & Santa Monica Blvd/UCLA,None,300,13:00:54,Southbound,6.995567,City of Santa Monica


In [4]:
# Set up a function that counts the number of 
# unique route_ids and route_type
def simple_route_aggregation(df: pd.DataFrame) -> pd.DataFrame:
        aggregated = (df.groupby(["calitp_itp_id",
                                  "organization_name"])
                      .agg({"route_id": "nunique", 
                            "route_type": "nunique"})
                      .reset_index()
                     )
        
        return aggregated


In [5]:
df_agg = simple_route_aggregation(df)

In [6]:
df_agg

,calitp_itp_id,organization_name,route_id,route_type
0,300,City of Santa Monica,18,1


### Move it to delayed

We can use the `@delayed` decorator right above our defined function.


Alternatively, you can wrap the function, like `delayed(my_function)(args)`. These are equivalent.

```
@delayed
def my_function(df):
    df2 = do something
    return df2
    
    
or...
delayed(my_function)(df)
```

**Note where the parentheses fall**...it is not a typo.


In [7]:
# We can use a decorator to make it a delayed function
@delayed
def import_data(itp_id: int):
    return pd.read_parquet(
        f"{GCS_FILE_PATH}{itp_id}_{analysis_date}.parquet")

In [8]:
# We have a list of 3 operators we would have looped over
operators

[182, 300, 282]

In [9]:
# Let's read in our data using list comprehension
dfs = [import_data(x) for x in operators]

In [10]:
# We have a list of delayed objects
# these dfs are not materialized / read into memory
dfs

[Delayed('import_data-2011297f-0ed9-452d-a5a7-3d53a2534273'),
 Delayed('import_data-8ad3a084-e0fd-47fc-b20b-f9155a02f89c'),
 Delayed('import_data-45f0f2dc-d6c5-48ea-a579-f5ef263023d4')]

In [11]:
# Set  up a list to store our results
results = [delayed(simple_route_aggregation)(df) for df in dfs]

In [12]:
# Wrap compute around each of the items in the results list 
# and see what's inside
results_computed = [compute(i) for i in results]

/tmp/ipykernel_572/3657125111.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  categorized = pd.Series(index=df.index)


In [13]:
# This is a list of tuples...that's not what we want
results_computed

[(   calitp_itp_id                                  organization_name  route_id  \
  0            182  Los Angeles County Metropolitan Transportation...       120   
  
     route_type  
  0           3  ,),
 (   calitp_itp_id     organization_name  route_id  route_type
  0            300  City of Santa Monica        18           1,),
 (   calitp_itp_id                 organization_name  route_id  route_type
  0            282  City and County of San Francisco        67           3,)]

In [14]:
type(results_computed[0])

tuple

In [15]:
# We need the first item of the tuple...that's our df
type(results_computed[0][0])

pandas.core.frame.DataFrame

In [16]:
results_computed_correct = [compute(i)[0] for i in results]

In [17]:
results_computed_correct

[   calitp_itp_id                                  organization_name  route_id  \
 0            182  Los Angeles County Metropolitan Transportation...       120   
 
    route_type  
 0           3  ,
    calitp_itp_id     organization_name  route_id  route_type
 0            300  City of Santa Monica        18           1,
    calitp_itp_id                 organization_name  route_id  route_type
 0            282  City and County of San Francisco        67           3]

In [18]:
type(results_computed_correct[0])

pandas.core.frame.DataFrame

In [19]:
results_computed_correct[0].head()

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3


In [20]:
# Alternatively, the code can be written like a loop, 
# but it won't run like a loop. It will run it simultaneously 
# for the three operators

results2 = []

for itp_id in operators:
    operator_df = import_data(itp_id)
    print(f"type for operator_df: {type(operator_df)}")
    
    aggregated_df = delayed(simple_route_aggregation)(operator_df)
    print(f"type for aggregated_df: {type(aggregated_df)}")
    
    results2.append(aggregated_df)


type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>
type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>
type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>


In [21]:
results_computed2 = [compute(i)[0] for i in results2]

In [22]:
results_computed2

[   calitp_itp_id                                  organization_name  route_id  \
 0            182  Los Angeles County Metropolitan Transportation...       120   
 
    route_type  
 0           3  ,
    calitp_itp_id     organization_name  route_id  route_type
 0            300  City of Santa Monica        18           1,
    calitp_itp_id                 organization_name  route_id  route_type
 0            282  City and County of San Francisco        67           3]

At this point, you can either write a function to export each individual aggregated pandas df result to be its standalone parquet, or combine it all. 

We will not export and overwrite the file in the GCS bucket right now.

Since our results are just pandas dfs, we could also concatenate them.

In [23]:
pd.concat(results_computed2, axis=0)

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3
0,300,City of Santa Monica,18,1
0,282,City and County of San Francisco,67,3


In [24]:
# This is rather pointless for such a small df, but for larger
# ones, we may want to concatenate and export it as a partitioned parquet
dd.multi.concat(results_computed2, axis=0).compute()

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3
0,300,City of Santa Monica,18,1
0,282,City and County of San Francisco,67,3


/tmp/ipykernel_572/3657125111.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  categorized = pd.Series(index=df.index)
/tmp/ipykernel_572/3657125111.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  categorized = pd.Series(index=df.index)


## To Do

* For the same 3 operators, use delayed functions throughout, from importing the parquet, applying a function, and saving the results to a list.
* Your function should group each trip into a category based on its `mean_speed_mph`. 
   * < 10 mph
   * 10-15 mph
   * 15-20 mph
   * 20+ mph
* For each operator, get the count of trips by category and its proportion
* Save the results in a list, compute the results for all the operators at once
* Concatenate the aggregated results for all the operators into one dask df

In [25]:
SPEED_CATEGORIES = {
    10.: "< 10 mph",
    15.: "10-15 mph",
    20.: "15-20 mph",
    np.inf: "20+ mph"
}
SPEED_COLUMN = "mean_speed_mph"
CATEGORY_COLUMN = "speed_category"
TRIP_ID_COLUMN = "trip_key"

@delayed
def import_trip_speed_df(operator_id: int, analysis_date_str: str):
    return pd.read_parquet(
        f"{GCS_FILE_PATH}{operator_id}_{analysis_date_str}.parquet"
    )

@delayed
def get_speed_category(df: pd.DataFrame, categories: dict) -> pd.DataFrame:
    categorized = pd.Series(index=df.index)
    for category_key in SPEED_CATEGORIES:
        # Using a loop like this that alters the same thing doesn't feel very
        # good for parallel processing
        # But since this is all one function anyway I'm not sure if it really matters?
        categorized.loc[
            categorized.isna() & (df[SPEED_COLUMN] < category_key)
        ] = SPEED_CATEGORIES[category_key]
    df_copy = df.copy()
    df_copy[CATEGORY_COLUMN]= categorized
    return df_copy

@delayed
def get_trip_count_by_category(df: pd.DataFrame, operator_name: object) -> pd.DataFrame:
    count_by_category = df.groupby(CATEGORY_COLUMN)[TRIP_ID_COLUMN].count().rename(
        f"{operator_name}_speed_category_count"
    )
    proportion_by_category = (
        count_by_category / count_by_category.sum()
    ).rename(
        f"{operator_name}_speed_category_proportion"
    )
    return pd.concat(
        [count_by_category, proportion_by_category], axis=1
    )
    

In [26]:
list_of_temp_dfs = [
    get_trip_count_by_category(
        get_speed_category(
            import_trip_speed_df(operator_id, analysis_date), SPEED_CATEGORIES
        ),
        operator_id
    )   
    for operator_id in operators
]
list_of_temp_dfs

[Delayed('get_trip_count_by_category-d0b099e7-a7c3-493f-a9b7-fb6e87ad3946'),
 Delayed('get_trip_count_by_category-7608b357-bfc8-4951-bda1-7151a103ea34'),
 Delayed('get_trip_count_by_category-70598d2b-659d-4f6a-bb9c-65a3f84a63f9')]

In [27]:
list_of_dfs = [compute(temp_df)[0] for temp_df in list_of_temp_dfs]
list_of_dfs

[                182_speed_category_count  182_speed_category_proportion
 speed_category                                                         
 10-15 mph                           5806                       0.392855
 15-20 mph                           1721                       0.116449
 20+ mph                             1511                       0.102240
 < 10 mph                            5741                       0.388457,
                 300_speed_category_count  300_speed_category_proportion
 speed_category                                                         
 10-15 mph                            497                       0.406710
 15-20 mph                             51                       0.041735
 20+ mph                                8                       0.006547
 < 10 mph                             666                       0.545008,
                 282_speed_category_count  282_speed_category_proportion
 speed_category                                  

In [30]:
ddf = dd.multi.concat(list_of_dfs, axis=1)
ddf.compute()

,182_speed_category_count,182_speed_category_proportion,300_speed_category_count,300_speed_category_proportion,282_speed_category_count,282_speed_category_proportion
speed_category,,,,,,
10-15 mph,5806,0.392855,497,0.406710,1451,0.161779
15-20 mph,1721,0.116449,51,0.041735,144,0.016055
20+ mph,1511,0.102240,8,0.006547,67,0.007470
< 10 mph,5741,0.388457,666,0.545008,7307,0.814695


In [ ]:
type(ddf)